In [ ]:
from AlgorithmImports import *
from QuantConnect.Research import QuantBook

from research_tools import ResearchTools

qb = QuantBook()

tools = ResearchTools(qb.ObjectStore)

records = tools.load(refresh=True)

print(f"Loaded {len(records)} experiment records")

In [ ]:
# Show all records

print(tools.show(records))

In [ ]:
# Top runs

top_runs = tools.top(
    records=records,
    sort_by="risk_adjusted_score",
    n=10
)

print(tools.show(top_runs))

In [ ]:
# Best survivors

best_survivors = tools.best_survivors(
    records=records,
    sort_by="risk_adjusted_score",
    n=10
)

print(tools.show(best_survivors))

In [ ]:
# Select BTC daily cohort

btc_daily = tools.select(
    coin="BTCUSD",
    timeframe="1d"
)

print(f"BTCUSD 1d cohort: {len(btc_daily)} records")

print(tools.show(tools.top(btc_daily, n=20)))

In [ ]:
# Select safe/profitable runs

safe_profitable = tools.select(
    ruined=False,
    net_profit__gt=0
)

print(f"Safe/profitable cohort: {len(safe_profitable)} records")
print(tools.show(tools.top(safe_profitable, n=10)))

In [ ]:
# Cohort summary by timeframe

summary_by_timeframe = tools.cohort_summary(
    records=records,
    group_by="timeframe"
)

print(tools.format_table(summary_by_timeframe))


In [ ]:
# Cohort summary by entry/filter

summary_by_entry_filter = tools.cohort_summary(
    records=records,
    group_by=["entry_model", "filter_model"]
)

print(tools.format_table(summary_by_entry_filter))


In [ ]:
# Compare top two runs

run_ids = [r["run_id"] for r in top_runs[:2]]

comparison = tools.compare(run_ids)

print(tools.format_table(comparison))

In [ ]:
# Load full report for top run

if len(top_runs) > 0:
    run_id = top_runs[0]["run_id"]
    report = tools.load_report(run_id)

    print(run_id)
    print(report.keys() if report is not None else "Report not found")
    

In [ ]:
# Optional DataFrame

df = tools.to_dataframe(records)
df.head()

In [ ]:
run_id = "BTCUSD_1d_candle_streak_martingale_20260710T190732Z_07a7a0eb0afd"

report = tools.load_report(run_id)

print(report.keys())
print(report["metadata"])

In [ ]:
records = tools.load(refresh=True)

v2_records = tools.select(record_source="v2")
legacy_records = tools.select(record_source="legacy")

print(f"Total records: {len(records)}")
print(f"v2 records: {len(v2_records)}")
print(f"legacy records: {len(legacy_records)}")

In [ ]:
print(tools.show(v2_records))

In [ ]:
import importlib

import report_models
import experiment_store
import research_tools

importlib.reload(report_models)
importlib.reload(experiment_store)
importlib.reload(research_tools)

from research_tools import ResearchTools

tools = ResearchTools(qb.ObjectStore)

records = tools.load(refresh=True)

v2_records = tools.select(record_source="v2")
legacy_records = tools.select(record_source="legacy")

print(f"Total records: {len(records)}")
print(f"v2 records: {len(v2_records)}")
print(f"legacy records: {len(legacy_records)}")

In [ ]:
# smoke test

from AlgorithmImports import *
from QuantConnect.Research import QuantBook

from research_tools import ResearchTools

qb = QuantBook()
tools = ResearchTools(qb.ObjectStore)

records = tools.load(refresh=True)

print(f"Total records: {len(records)}")
print(f"v2 records: {len(tools.select(record_source='v2'))}")
print(f"legacy records: {len(tools.select(record_source='legacy'))}")

In [ ]:
print(tools.show(tools.select(record_source="v2")))

In [ ]:
import importlib

import research_tools
importlib.invalidate_caches()
importlib.reload(research_tools)

from research_tools import ResearchTools

tools = ResearchTools(qb.ObjectStore)
records = tools.load(refresh=True)

print(f"Total records: {len(records)}")
print(f"v2 records: {len(tools.v2())}")
print(f"legacy records: {len(tools.legacy())}")

In [ ]:
print(tools.dashboard_text(top_n=5))

In [ ]:
daily_candidates = tools.select(
    coin="BTCUSD",
    timeframe="1d",
    entry_model="candle_streak",
    filter_model="ema_trend_20_50",
    stake_mode="martingale"
)

print(f"Candidates: {len(daily_candidates)}")
print(tools.show(daily_candidates, n=20))

for r in daily_candidates:
    print()
    print("run_id:", r.get("run_id"))
    print("record_source:", r.get("record_source"))
    print("start:", r.get("start"), type(r.get("start")))
    print("end:", r.get("end"), type(r.get("end")))
    print("streak_length:", r.get("streak_length"), type(r.get("streak_length")))
    print("streak_mode:", r.get("streak_mode"), type(r.get("streak_mode")))
    print("ema_fast:", r.get("ema_fast"), type(r.get("ema_fast")))
    print("ema_slow:", r.get("ema_slow"), type(r.get("ema_slow")))
    print("base_wager:", r.get("base_wager"), type(r.get("base_wager")))
    print("bankroll:", r.get("bankroll"), type(r.get("bankroll")))
    print("multiplier:", r.get("multiplier"), type(r.get("multiplier")))
    print("max_steps:", r.get("max_steps"), type(r.get("max_steps")))
    print("signature:", tools.config_signature(r))

In [ ]:
import importlib
import research_tools

importlib.invalidate_caches()
importlib.reload(research_tools)

from research_tools import ResearchTools

tools = ResearchTools(qb.ObjectStore)
records = tools.load(refresh=True)

dupes = tools.duplicates()
deduped = tools.dedupe()

print(f"Original records: {len(records)}")
print(f"Deduped records: {len(deduped)}")
print(f"Duplicate config groups: {len(dupes)}")

if len(dupes) > 0:
    print(tools.show(dupes[0]["records"]))

In [ ]:
latest_v2 = tools.latest_v2(n=1)

print(tools.show(latest_v2))

if latest_v2:
    run_id = latest_v2[0]["run_id"]
    record = tools.get(run_id)
    report = tools.load_report(run_id)

    print("\nNORMALIZED RECORD METADATA")
    for field in [
        "lab_version",
        "experiment_id",
        "experiment_group",
        "experiment_name",
        "experiment_notes",
        "experiment_tags",
        "experiment_year",
        "experiment_asset",
        "experiment_timeframe",
        "experiment_family"
    ]:
        print(f"{field}: {record.get(field)}")

    print("\nFULL REPORT METADATA")
    for key, value in report["metadata"].items():
        print(f"{key}: {value}")

In [ ]:
tools.select(experiment_id="exp_0030_btcusd_1d_2026_ema_streak_follow")

In [ ]:
tools.select(experiment_group="btcusd_2026")

In [ ]:
tools.select(experiment_family="ema_streak_follow")

In [ ]:
tools.select(experiment_tags__contains="candle_streak")

In [ ]:
tools = ResearchTools(qb.ObjectStore)

# Force complete reload
records = tools.load(refresh=True)

print(f"Total records after refresh: {len(records)}")

# Show the 15 most recent runs (by generation time)
latest = tools.latest(n=15)
print(tools.show(latest))

# Specifically look for any finrev runs that have the new field
finrev_runs = tools.select(entry_model="finrev")
print(f"\nFinRev runs found: {len(finrev_runs)}")
print(tools.show(tools.sort(finrev_runs, sort_by="generated_at_utc", reverse=True), n=20))

In [ ]:
records = tools.load(refresh=True)
finrev = tools.select(entry_model="finrev")
print(tools.show(finrev, fields=[
    "run_id", "timeframe", "finrev_threshold", "trades", "win_rate", "net_profit"
], n=20))

In [31]:
from AlgorithmImports import *
from QuantConnect.Research import QuantBook

qb = QuantBook()
store = qb.ObjectStore


def get_keys(object_store):
    keys = getattr(object_store, "Keys", None)

    if keys is None:
        keys = getattr(object_store, "keys", None)

    if callable(keys):
        keys = keys()

    if keys is not None:
        return [str(key) for key in keys]

    output = []

    for item in object_store:
        key = getattr(item, "Key", None)

        if key is None:
            key = getattr(item, "key", None)

        if key is not None:
            output.append(str(key))

    return output


keys = get_keys(store)

max_files = getattr(
    store,
    "MaxFiles",
    getattr(store, "max_files", None)
)

print(f"ObjectStore files: {len(keys)}")
print(f"Maximum files: {max_files}")

prefixes = [
    "qcrl/results/",
    "qcrl/v2/experiments/",
    "qcrl/v2/records/"
]

for prefix in prefixes:
    matching = [key for key in keys if key.startswith(prefix)]
    print(f"{prefix}: {len(matching)}")

In [32]:
import json


def contains_key(object_store, key):
    method = getattr(
        object_store,
        "ContainsKey",
        getattr(object_store, "contains_key", None)
    )

    return bool(method(key))


def read_text(object_store, key):
    method = getattr(
        object_store,
        "Read",
        getattr(object_store, "read", None)
    )

    return method(key)


record_keys = [
    key
    for key in get_keys(store)
    if key.startswith("qcrl/v2/records/")
]

mirrored_record_keys = []
orphan_record_keys = []

for record_key in record_keys:
    try:
        record = json.loads(read_text(store, record_key))
        report_key = record.get("objectstore_report_key")

        if report_key and contains_key(store, report_key):
            mirrored_record_keys.append(record_key)
        else:
            orphan_record_keys.append(record_key)

    except Exception as error:
        print(f"Could not inspect {record_key}: {error}")

print(f"Mirrored records safe to delete: {len(mirrored_record_keys)}")
print(f"Orphan records to preserve: {len(orphan_record_keys)}")

In [33]:
delete_method = getattr(
    store,
    "Delete",
    getattr(store, "delete", None)
)

deleted = 0

for key in mirrored_record_keys:
    if contains_key(store, key):
        delete_method(key)
        deleted += 1

clear_method = getattr(
    store,
    "Clear",
    getattr(store, "clear", None)
)

if clear_method is not None:
    clear_method()

print(f"Deleted redundant normalized records: {deleted}")